# KarirFit — Model Rekomendasi Karier IT

Notebook ini adalah *offline pipeline* dari skripsi (Bab III): mengubah data lowongan kerja IT Indonesia menjadi model skor karier, mengevaluasinya, lalu mengekspor `model.json` / `model.js` untuk website.

| Bagian | Isi | Bagian skripsi |
|---|---|---|
| 1–2 | Konfigurasi dan katalog (kamus skill, 15 karier, 9 minat, mata kuliah) | 3.2.1, 3.5.2 |
| 3 | Memuat dataset lowongan | 3.2.2, Tabel 3.4 |
| 4 | Klasifikasi judul lowongan ke karier + sampel validasi | 3.2.2, 3.2.5 |
| 5 | Ekstraksi skill dari deskripsi + sampel validasi | 3.2.2, 3.2.5 |
| 6 | Split 70/30 bertingkat per karier | 3.2.2 |
| 7 | Permintaan d(c,s), himpunan kebutuhan R(c), bobot w(c,s) | Pers. 3.1–3.2 |
| 7b | Triangulasi dengan SKKNI/peta okupasi dan O\*NET | 3.2.2, 3.2.5 |
| 8 | Cakupan kurikulum berbobot permintaan | RQ1 |
| 9 | Mekanisme skor | Pers. 3.3–3.5 |
| 10 | Evaluasi Hit@1, Hit@3, MRR vs baseline cosine | 3.6.2, RQ4 |
| 11 | Sensitivitas α dan ablasi | 3.6.3 |
| 12 | Coba profil | — |
| 13 | Ekspor model untuk website | 3.5.5 |
| 14 | (Opsional) indeks RAG | 3.2.4, 3.5.4 |

Jalankan dari atas ke bawah: **Runtime → Run all**. Pilih sumber data di bagian 3.

## 1. Konfigurasi

In [ ]:
import re, json, math, random, csv, io, os
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42             # seed acak tetap (split 70/30 dan sampel validasi)
THRESHOLD = 0.10      # skill masuk R(c) jika diminta di >= 10% lowongan karier c
TEST_FRAC = 0.30      # porsi lowongan yang ditahan untuk pengujian
MIN_SKILLS_TEST = 3   # profil uji minimal punya 3 skill dari kamus
ALPHA = 0.5           # bobot kecocokan kompetensi pada Fit(c)
GRADE_POINTS = {"A": 1.00, "B": 0.75, "C": 0.50, "D": 0.0, "E": 0.0}
SELF_LEVEL = {"advanced": 1.00, "intermediate": 0.67, "beginner": 0.33}
OUT_DIR = "karirfit_output"
os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

## 2. Katalog

**Kamus skill (D6).** Setiap skill punya pola regex yang dicocokkan ke teks huruf kecil. Tambah atau perbaiki pola di sini jika hasil validasi ekstraksi (bagian 5) masih rendah.

In [ ]:
SKILLS = [
    # Programming languages
    ("python", "Python", "Programming", [r"\bpython\b"]),
    ("java", "Java", "Programming", [r"\bjava\b(?!\s*script)"]),
    ("javascript", "JavaScript", "Programming", [r"\bjavascript\b", r"\bjs\b", r"\bes6\b"]),
    ("typescript", "TypeScript", "Programming", [r"\btypescript\b"]),
    ("php", "PHP", "Programming", [r"\bphp\b"]),
    ("csharp", "C#", "Programming", [r"c#", r"\bc sharp\b"]),
    ("cpp", "C/C++", "Programming", [r"c\+\+", r"\bbahasa c\b", r"\bc language\b"]),
    ("golang", "Go (Golang)", "Programming", [r"\bgolang\b", r"\bgo (?:language|lang|programming)\b"]),
    ("kotlin", "Kotlin", "Programming", [r"\bkotlin\b"]),
    ("swift", "Swift", "Programming", [r"\bswift\b(?!\s*code)"]),
    ("dart", "Dart", "Programming", [r"\bdart\b"]),
    ("r_lang", "R language", "Programming", [r"\br (?:programming|language|studio|statistical)\b", r"\brstudio\b", r"\bbahasa r\b"]),
    ("oop", "Object-Oriented Programming", "Programming", [r"\boop\b", r"object[- ]oriented", r"berorientasi objek"]),
    ("data_structures", "Data Structures & Algorithms", "Programming", [r"data structures?", r"struktur data", r"\balgorithms?\b", r"\balgoritma\b"]),
    ("git", "Git / Version Control", "Programming", [r"\bgit\b", r"\bgithub\b", r"\bgitlab\b", r"version control"]),
    ("unit_testing", "Unit Testing", "Programming", [r"unit test", r"\bjunit\b", r"\bphpunit\b", r"\bjest\b", r"\bpytest\b"]),
    # Web
    ("html_css", "HTML/CSS", "Web", [r"\bhtml5?\b", r"\bcss3?\b"]),
    ("css_framework", "Bootstrap/Tailwind", "Web", [r"\bbootstrap\b", r"\btailwind\b"]),
    ("react", "React", "Web", [r"\breact(?:\.?js)?\b(?!\s*native)"]),
    ("vue", "Vue.js", "Web", [r"\bvue(?:\.?js)?\b", r"\bnuxt\b"]),
    ("angular", "Angular", "Web", [r"\bangular(?:js)?\b"]),
    ("nextjs", "Next.js", "Web", [r"\bnext\.?js\b"]),
    ("nodejs", "Node.js", "Web", [r"\bnode\.?js\b", r"\bexpress(?:\.?js)?\b", r"\bnestjs\b"]),
    ("laravel", "Laravel", "Web", [r"\blaravel\b"]),
    ("codeigniter", "CodeIgniter", "Web", [r"\bcodeigniter\b", r"\bci4?\b"]),
    ("spring", "Spring Boot", "Web", [r"\bspring(?: boot)?\b"]),
    ("django", "Django", "Web", [r"\bdjango\b"]),
    ("flask", "Flask/FastAPI", "Web", [r"\bflask\b", r"\bfastapi\b"]),
    ("dotnet", ".NET", "Web", [r"\.net\b", r"\basp\.net\b", r"\bdotnet\b"]),
    ("rest_api", "REST API", "Web", [r"\brest(?:ful)?\b", r"\bapi\b", r"web service"]),
    ("graphql", "GraphQL", "Web", [r"\bgraphql\b"]),
    ("microservices", "Microservices", "Web", [r"micro-?services?"]),
    # Mobile
    ("android", "Android", "Mobile", [r"\bandroid\b"]),
    ("ios", "iOS", "Mobile", [r"\bios\b", r"\bxcode\b"]),
    ("flutter", "Flutter", "Mobile", [r"\bflutter\b"]),
    ("react_native", "React Native", "Mobile", [r"react native"]),
    # Data & databases
    ("sql", "SQL", "Data", [r"\bsql\b", r"\bquery\b"]),
    ("mysql", "MySQL", "Data", [r"\bmysql\b", r"\bmariadb\b"]),
    ("postgresql", "PostgreSQL", "Data", [r"\bpostgre(?:s|sql)?\b"]),
    ("sql_server", "SQL Server", "Data", [r"sql server", r"\bmssql\b", r"\bt-sql\b"]),
    ("oracle_db", "Oracle Database", "Data", [r"\boracle\b", r"\bpl/sql\b"]),
    ("mongodb", "MongoDB", "Data", [r"\bmongo(?:db)?\b"]),
    ("redis", "Redis", "Data", [r"\bredis\b"]),
    ("nosql", "NoSQL", "Data", [r"\bnosql\b", r"\bcassandra\b", r"\bfirebase\b"]),
    ("backup_recovery", "Backup & Recovery", "Data", [r"\bbackup\b", r"disaster recovery", r"\brestore\b"]),
    ("db_tuning", "Database Performance Tuning", "Data", [r"performance tuning", r"query optimi[sz]ation", r"\bindexing\b"]),
    ("data_modeling", "Data Modeling / ERD", "Data", [r"data model(?:l)?ing", r"\berd\b", r"entity relationship", r"normali[sz]ation"]),
    ("etl", "ETL / Data Pipeline", "Data", [r"\betl\b", r"\belt\b", r"data pipelines?"]),
    ("data_warehouse", "Data Warehouse", "Data", [r"data warehous", r"\bbigquery\b", r"\bsnowflake\b", r"\bredshift\b"]),
    ("spark", "Apache Spark", "Data", [r"\bspark\b", r"\bpyspark\b"]),
    ("hadoop", "Hadoop / Big Data", "Data", [r"\bhadoop\b", r"big data", r"\bhive\b"]),
    ("airflow", "Airflow", "Data", [r"\bairflow\b"]),
    ("kafka", "Kafka", "Data", [r"\bkafka\b"]),
    ("excel", "Excel / Spreadsheet", "Data", [r"\bexcel\b", r"spreadsheet", r"google sheets?"]),
    ("power_bi", "Power BI", "Data", [r"power ?bi\b"]),
    ("tableau", "Tableau", "Data", [r"\btableau\b"]),
    ("looker", "Looker / Data Studio", "Data", [r"\blooker\b", r"data studio"]),
    ("statistics", "Statistics", "Data", [r"\bstatistic", r"\bstatistik"]),
    ("data_visualization", "Data Visualization", "Data", [r"data visuali[sz]ation", r"visualisasi data", r"\bdashboards?\b"]),
    ("pandas", "Pandas / NumPy", "Data", [r"\bpandas\b", r"\bnumpy\b"]),
    ("machine_learning", "Machine Learning", "AI", [r"machine learning", r"\bml\b", r"pembelajaran mesin"]),
    ("deep_learning", "Deep Learning", "AI", [r"deep learning", r"neural networks?", r"\bcnn\b", r"\blstm\b"]),
    ("tensorflow", "TensorFlow/Keras", "AI", [r"\btensorflow\b", r"\bkeras\b"]),
    ("pytorch", "PyTorch", "AI", [r"\bpytorch\b"]),
    ("scikit_learn", "scikit-learn", "AI", [r"scikit[- ]learn", r"\bsklearn\b"]),
    ("nlp", "NLP", "AI", [r"\bnlp\b", r"natural language"]),
    ("computer_vision", "Computer Vision", "AI", [r"computer vision", r"\bopencv\b", r"image processing", r"pengolahan citra"]),
    ("gen_ai", "Generative AI / LLM", "AI", [r"\bllms?\b", r"generative ai", r"\bgenai\b", r"\bgpt\b", r"\brag\b"]),
    # Infrastructure & cloud
    ("linux", "Linux", "Infrastructure", [r"\blinux\b", r"\bubuntu\b", r"\bcentos\b", r"\bdebian\b", r"\bred ?hat\b"]),
    ("windows_server", "Windows Server", "Infrastructure", [r"windows server"]),
    ("active_directory", "Active Directory", "Infrastructure", [r"active directory", r"\bldap\b"]),
    ("networking", "Computer Networking (TCP/IP)", "Infrastructure", [r"\bnetwork(?:ing)?\b", r"\btcp/?ip\b", r"\blan\b", r"\bwan\b", r"jaringan komputer", r"\brouting\b", r"\bswitching\b"]),
    ("cisco", "Cisco / CCNA", "Infrastructure", [r"\bcisco\b", r"\bccna\b", r"\bccnp\b"]),
    ("mikrotik", "MikroTik", "Infrastructure", [r"\bmikrotik\b", r"\bmtcna\b"]),
    ("virtualization", "Virtualization (VMware)", "Infrastructure", [r"\bvmware\b", r"virtuali[sz]ation", r"\bhyper-v\b", r"\bproxmox\b"]),
    ("docker", "Docker", "Infrastructure", [r"\bdocker\b", r"\bcontainers?\b"]),
    ("kubernetes", "Kubernetes", "Infrastructure", [r"\bkubernetes\b", r"\bk8s\b", r"\bopenshift\b"]),
    ("aws", "AWS", "Infrastructure", [r"\baws\b", r"amazon web services", r"\bec2\b"]),
    ("gcp", "Google Cloud", "Infrastructure", [r"\bgcp\b", r"google cloud"]),
    ("azure", "Microsoft Azure", "Infrastructure", [r"\bazure\b"]),
    ("ci_cd", "CI/CD", "Infrastructure", [r"\bci/?cd\b", r"continuous integration", r"continuous delivery", r"github actions", r"gitlab ci"]),
    ("jenkins", "Jenkins", "Infrastructure", [r"\bjenkins\b"]),
    ("terraform", "Terraform / IaC", "Infrastructure", [r"\bterraform\b", r"infrastructure as code", r"\biac\b"]),
    ("ansible", "Ansible", "Infrastructure", [r"\bansible\b", r"\bpuppet\b", r"\bchef\b"]),
    ("monitoring", "Monitoring (Prometheus/Grafana)", "Infrastructure", [r"\bprometheus\b", r"\bgrafana\b", r"\bzabbix\b", r"\bnagios\b", r"\bmonitoring\b"]),
    ("scripting", "Shell Scripting", "Infrastructure", [r"\bbash\b", r"shell script", r"\bpowershell\b"]),
    # Security
    ("network_security", "Network Security", "Security", [r"network security", r"keamanan jaringan", r"\bvpn\b", r"\bids\b", r"\bips\b"]),
    ("firewall", "Firewall", "Security", [r"\bfirewalls?\b", r"\bfortigate\b", r"palo alto"]),
    ("pentest", "Penetration Testing", "Security", [r"penetration test", r"\bpentest", r"ethical hack", r"\bburp\b", r"\bkali\b"]),
    ("vuln_assessment", "Vulnerability Assessment", "Security", [r"vulnerabilit", r"\bvapt\b", r"\bowasp\b", r"\bnessus\b"]),
    ("siem", "SIEM / SOC", "Security", [r"\bsiem\b", r"\bsoc\b", r"\bsplunk\b", r"\bqradar\b", r"\bwazuh\b"]),
    ("iso27001", "ISO 27001 / Security Governance", "Security", [r"iso ?27001", r"\bnist\b", r"risk assessment", r"\bisms\b"]),
    # QA
    ("manual_testing", "Manual Testing", "QA", [r"manual test", r"black ?box", r"\buat\b", r"functional test"]),
    ("test_case", "Test Case / Test Plan", "QA", [r"test cases?", r"test plan", r"test scenario", r"bug report"]),
    ("automation_testing", "Test Automation", "QA", [r"automation test", r"automated test", r"test automation"]),
    ("selenium", "Selenium / Cypress", "QA", [r"\bselenium\b", r"\bcypress\b", r"\bappium\b", r"\bkatalon\b", r"\bplaywright\b"]),
    ("api_testing", "API Testing (Postman)", "QA", [r"\bpostman\b", r"api test"]),
    ("performance_testing", "Performance Testing (JMeter)", "QA", [r"\bjmeter\b", r"performance test", r"load test", r"\bk6\b"]),
    # Design
    ("figma", "Figma", "Design", [r"\bfigma\b"]),
    ("ui_design", "UI Design", "Design", [r"\bui\b", r"user interface", r"visual design"]),
    ("ux_research", "UX Research", "Design", [r"\bux\b", r"user research", r"usability", r"user experience"]),
    ("prototyping", "Wireframing & Prototyping", "Design", [r"\bwireframes?\b", r"\bprototyp", r"\bmockups?\b"]),
    ("adobe", "Adobe XD/Photoshop/Illustrator", "Design", [r"adobe", r"photoshop", r"illustrator", r"\bxd\b"]),
    ("design_system", "Design System", "Design", [r"design system", r"style guide"]),
    # Analysis & management
    ("requirement_analysis", "Requirement Analysis", "Analysis", [r"requirements? (?:analysis|gathering|elicitation)", r"analisis kebutuhan", r"business requirements?", r"\bbrd\b"]),
    ("uml", "UML", "Analysis", [r"\buml\b", r"use case", r"sequence diagram", r"class diagram"]),
    ("bpmn", "Business Process Modeling (BPMN)", "Analysis", [r"\bbpmn\b", r"business process", r"proses bisnis", r"\bflowchart\b"]),
    ("sdlc", "SDLC", "Analysis", [r"\bsdlc\b", r"software development life ?cycle", r"\bwaterfall\b"]),
    ("agile_scrum", "Agile / Scrum", "Analysis", [r"\bagile\b", r"\bscrum\b", r"\bkanban\b", r"\bsprint\b"]),
    ("jira", "Jira / Trello", "Analysis", [r"\bjira\b", r"\btrello\b", r"\bconfluence\b"]),
    ("documentation", "Technical Documentation", "Analysis", [r"documentation", r"dokumentasi", r"\bsrs\b", r"technical writing"]),
    ("project_management", "Project Management", "Management", [r"project manage", r"manajemen proyek", r"\bpmp\b", r"\bprince2\b"]),
    ("product_management", "Product Management", "Management", [r"product manage", r"product roadmap", r"\bprd\b", r"product owner"]),
    ("erp", "ERP (SAP/Odoo)", "Management", [r"\berp\b", r"\bsap\b", r"\bodoo\b"]),
    ("itil", "ITIL / IT Service Management", "Management", [r"\bitil\b", r"service management", r"\bitsm\b", r"\bsla\b"]),
    # IT support
    ("troubleshooting", "Troubleshooting", "Support", [r"troubleshoot", r"\bperbaikan\b"]),
    ("hardware", "Hardware & PC Maintenance", "Support", [r"\bhardware\b", r"perangkat keras", r"\bprinters?\b", r"pc maintenance", r"\bmaintenance\b"]),
    ("helpdesk", "Helpdesk / Ticketing", "Support", [r"help ?desk", r"service desk", r"\bticketing\b", r"\bfreshdesk\b", r"\bzendesk\b"]),
    ("os_install", "OS Installation & Configuration", "Support", [r"\bos install", r"install(?:ation|asi)? (?:os|windows|software|sistem operasi)", r"\bwindows\b(?! server)"]),
    ("ms_office", "Microsoft Office", "Support", [r"ms office", r"microsoft office", r"\bword\b", r"\bpowerpoint\b", r"\boutlook\b"]),
]

# (id, name_en, name_id, group, title regex patterns used to classify job ads)

**Karier dan bidang minat.** Pola judul dipakai untuk mengklasifikasikan lowongan. Senior Programmer dipisahkan dari Software Developer lewat kata *senior / lead / principal / sr.* pada judul (3.2.2). Bidang minat mengikuti SKKNI dan peta okupasi TIK (3.5.2).

In [ ]:
CAREERS = [
    ("senior_programmer", "Senior Programmer", "Senior Programmer", "primary",
     [r"\b(?:senior|sr\.?|lead|principal)\b.*\b(?:programmer|(?:software|web|back-?end|front-?end|full-?stack|java|php|python|\.net|node(?:\.?js)?|golang|ruby|application|app)\s+(?:developer|engineer)|developer)\b",
      r"\btech(?:nical)? lead\b"]),
    ("software_developer", "Software Developer", "Pengembang Perangkat Lunak", "primary",
     [r"\b(?:programmer|software (?:developer|engineer)|web developer|back-?end|front-?end|full-?stack)\b"]),
    ("system_analyst", "System Analyst", "Analis Sistem", "primary",
     [r"\b(?:system|systems|it|business system) analyst\b", r"\banalis sistem\b"]),
    ("database_administrator", "Database Administrator", "Administrator Basis Data", "primary",
     [r"\b(?:database administrator|dba|database engineer)\b"]),
    ("mobile_developer", "Mobile Developer", "Pengembang Aplikasi Mobile", "alternative",
     [r"\b(?:mobile|android|ios|flutter|react native) (?:developer|engineer|programmer)\b"]),
    ("qa_tester", "QA / Software Tester", "QA / Penguji Perangkat Lunak", "alternative",
     [r"\b(?:qa|sdet|software tester|sw tester|software test|software quality|test automation|automation test|quality assurance (?:engineer|analyst|tester|staff))\b",
      r"^(?:junior |senior |sr\.? )?(?:tester|test engineer)\b"]),
    ("ui_ux_designer", "UI/UX Designer", "Desainer UI/UX", "alternative",
     [r"\b(?:ui|ux|ui/ux|product designer|user experience|user interface)\b"]),
    ("data_analyst", "Data Analyst / BI", "Analis Data / BI", "alternative",
     [r"\b(?:data analyst|business intelligence|bi (?:analyst|developer)|analis data)\b"]),
    ("data_engineer", "Data Engineer", "Data Engineer", "alternative",
     [r"\b(?:data engineer|big data engineer|etl developer)\b"]),
    ("data_scientist", "Data Scientist / ML / AI Engineer", "Data Scientist / ML / AI Engineer", "alternative",
     [r"\b(?:data scientist|machine learning|ml engineer|ai engineer|deep learning)\b"]),
    ("devops_cloud", "DevOps / Cloud Engineer", "DevOps / Cloud Engineer", "alternative",
     [r"\b(?:devops|cloud engineer|site reliability|sre|platform engineer)\b"]),
    ("network_sysadmin", "Network / System Administrator", "Administrator Jaringan / Sistem", "alternative",
     [r"\b(?:network (?:engineer|administrator)|system administrator|sysadmin|admin jaringan|noc)\b"]),
    ("cyber_security", "Cyber Security", "Keamanan Siber", "alternative",
     [r"\b(?:security (?:analyst|engineer)|cyber ?security|penetration tester|soc analyst|infosec)\b"]),
    ("it_support", "IT Support / Helpdesk", "IT Support / Helpdesk", "alternative",
     [r"\b(?:it support|it helpdesk|helpdesk|help desk|service desk|desktop support|it technical support|it staff|staff it)\b"]),
    ("it_pm", "IT Project / Product Manager", "Manajer Proyek / Produk TI", "alternative",
     [r"\b(?:it|software|digital|tech|technology|erp|sap|system|systems|web|app|data|cloud)\b.*\b(?:project|program|delivery) manager\b",
      r"\b(?:product manager|product owner|scrum master|it manager|it project)\b"]),
]

INTERESTS = [
    ("software_dev", "Software development", "Pengembangan perangkat lunak", ["senior_programmer", "software_developer", "mobile_developer"]),
    ("systems_analysis", "Systems analysis", "Analisis sistem", ["system_analyst"]),
    ("data_mgmt", "Databases & data management", "Basis data & manajemen data", ["database_administrator", "data_engineer"]),
    ("data_ai", "Data science & AI", "Sains data & AI", ["data_analyst", "data_engineer", "data_scientist"]),
    ("infra_cloud", "Infrastructure & cloud", "Infrastruktur & cloud", ["network_sysadmin", "devops_cloud"]),
    ("security", "Security", "Keamanan", ["cyber_security"]),
    ("qa", "Quality assurance", "Penjaminan mutu (QA)", ["qa_tester"]),
    ("design", "Design (UI/UX)", "Desain (UI/UX)", ["ui_ux_designer"]),
    ("it_service_mgmt", "IT service & management", "Layanan & manajemen TI", ["it_support", "it_pm"]),
]

# Titles containing these words are never IT careers (mixed-industry datasets: manufacturing,
# sales, logistics ...). Checked before the career patterns. Extend if validation shows leaks.
NON_IT_TITLE = [
    r"\bsupplier\b", r"\bmechanical\b", r"\bmanufactur", r"\bassembl", r"\bwelding\b", r"\bchassis\b",
    r"\bsales\b", r"\blogistic", r"\bhse\b", r"\bwarehouse\b", r"\bprocurement\b", r"\bpurchas",
    r"\bmarketing\b", r"\baccount(?:ing|ant)\b", r"\bfinance\b", r"\bhr\b", r"\bhuman resource",
    r"\bprocess engineer", r"\bquality engineer", r"\bfacility", r"\bmaintenance technician",
    r"\belectrical\b", r"\bhardware\b", r"\bemotor\b", r"\bebike\b", r"\bplant\b",
]

# Example course -> skill map for the public version (common Indonesian IT courses).
# Replace with the 62 RPS-verified pairs from Section 3.2.3 for the PNJ deployment.

**Mata kuliah → skill.** Ini contoh mata kuliah umum di program studi IT Indonesia untuk versi publik. Untuk uji di PNJ, ganti dengan 62 pasangan mata kuliah–skill yang sudah diverifikasi dari RPS (3.2.3).

In [ ]:
COURSES = [
    ("algo", "Algoritma & Pemrograman", "Algorithms & Programming", ["data_structures", "python", "cpp"]),
    ("oop", "Pemrograman Berorientasi Objek", "Object-Oriented Programming", ["oop", "java"]),
    ("struktur_data", "Struktur Data", "Data Structures", ["data_structures"]),
    ("basis_data", "Basis Data", "Databases", ["sql", "mysql", "data_modeling"]),
    ("basis_data_lanjut", "Basis Data Lanjut", "Advanced Databases", ["sql", "postgresql", "db_tuning", "backup_recovery"]),
    ("web", "Pemrograman Web", "Web Programming", ["html_css", "javascript", "php", "css_framework"]),
    ("web_lanjut", "Pemrograman Web Lanjut", "Advanced Web Programming", ["laravel", "rest_api", "git", "mysql"]),
    ("mobile", "Pemrograman Mobile", "Mobile Programming", ["android", "kotlin", "flutter", "dart"]),
    ("rpl", "Rekayasa Perangkat Lunak", "Software Engineering", ["sdlc", "uml", "requirement_analysis", "agile_scrum", "unit_testing"]),
    ("apsi", "Analisis & Perancangan Sistem", "Systems Analysis & Design", ["requirement_analysis", "uml", "bpmn", "documentation"]),
    ("imk", "Interaksi Manusia & Komputer", "Human-Computer Interaction", ["ui_design", "ux_research", "prototyping", "figma"]),
    ("jarkom", "Jaringan Komputer", "Computer Networks", ["networking", "cisco", "mikrotik"]),
    ("so", "Sistem Operasi", "Operating Systems", ["linux", "scripting"]),
    ("keamanan", "Keamanan Informasi", "Information Security", ["network_security", "vuln_assessment", "firewall"]),
    ("cloud", "Komputasi Awan", "Cloud Computing", ["docker", "aws", "virtualization"]),
    ("statistika", "Statistika", "Statistics", ["statistics", "excel"]),
    ("data_mining", "Data Mining", "Data Mining", ["machine_learning", "python", "pandas", "scikit_learn"]),
    ("ai", "Kecerdasan Buatan", "Artificial Intelligence", ["machine_learning", "deep_learning", "python"]),
    ("bigdata", "Big Data", "Big Data", ["hadoop", "spark", "etl"]),
    ("testing", "Pengujian Perangkat Lunak", "Software Testing", ["manual_testing", "test_case", "automation_testing", "selenium"]),
    ("manpro", "Manajemen Proyek TI", "IT Project Management", ["project_management", "agile_scrum", "jira"]),
    ("orkom", "Organisasi & Arsitektur Komputer", "Computer Organisation", ["hardware"]),
    ("vis_data", "Visualisasi Data", "Data Visualization", ["data_visualization", "power_bi", "tableau"]),
]

### Pakai kamus skill (D6) dan peta mata kuliah–skill (D7) milikmu

Daftar `SKILLS` dan `COURSES` di atas adalah **contoh**. Untuk skripsi, ganti dengan file milikmu (Tabel 3.2):

- **D6** (`kamus_skill.csv`): 109 skill. Kolom yang dikenali: `skill` / `nama` (wajib), `id` (opsional), `category` / `kategori`, dan `patterns` / `pola` / `keywords` (beberapa pola dipisah `|`; pakai `;` hanya kalau pemisah kolom file-nya koma). Pola boleh berupa regex (`\bsql\b`) atau kata biasa (`sql server`); kata biasa diubah otomatis menjadi regex.
- **D7** (`peta_mk_skill.csv`): 62 pasangan. Satu baris per pasangan, dengan kolom `course` / `mata_kuliah` (wajib), `skill` (wajib, berisi id atau nama skill dari D6), `course_en` (opsional), dan `evidence` / `bukti` (opsional, kalimat pendukung dari RPS).

Atur `D6_MODE` / `D7_MODE`: `"upload"`, `"drive"` (isi path-nya), atau `"contoh"` untuk tetap memakai daftar di atas.

In [ ]:
D6_MODE, D6_DRIVE_PATH = "contoh", "/content/drive/MyDrive/skripsi/data/kamus_skill.csv"
D7_MODE, D7_DRIVE_PATH = "contoh", "/content/drive/MyDrive/skripsi/data/peta_mk_skill.csv"

def _slug(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")

def _col(df, *names):
    low = {c.lower().strip().replace(" ", "_"): c for c in df.columns}
    for n in names:
        if n in low:
            return low[n]
    return None

def _read_table(mode, path, label):
    if mode == "contoh":
        return None
    if mode == "upload":
        from google.colab import files
        print(f"Unggah {label}:")
        name, data = next(iter(files.upload().items()))
    elif mode == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        name, data = os.path.basename(path), open(path, "rb").read()
    else:
        raise ValueError("mode harus 'upload', 'drive', atau 'contoh'")
    if name.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(io.BytesIO(data))
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(io.BytesIO(data), encoding=enc, sep=None, engine="python")
        except UnicodeDecodeError:
            pass
    raise ValueError(f"{name}: encoding tidak dikenali")

def _to_regex(p):
    p = str(p).strip()
    if not p:
        return None
    if re.search(r"\\[bdsw]|[\[\](){}^$*+?|]", p):      # sudah berbentuk regex
        return p
    return r"\b" + re.escape(p.lower()) + r"\b"

d6 = _read_table(D6_MODE, D6_DRIVE_PATH, "D6 kamus skill")
if d6 is not None:
    c_name = _col(d6, "skill", "nama", "name", "skill_name", "nama_skill")
    c_id, c_cat = _col(d6, "id", "skill_id"), _col(d6, "category", "kategori", "categories")
    c_pat = _col(d6, "patterns", "pattern", "pola", "keywords", "keyword", "regex", "sinonim", "synonyms")
    assert c_name, f"Kolom nama skill tidak ditemukan. Kolom D6: {list(d6.columns)}"
    new_skills, seen = [], set()
    for _, r in d6.iterrows():
        name = str(r[c_name]).strip()
        if not name or name.lower() == "nan":
            continue
        sid = _slug(r[c_id]) if c_id and str(r[c_id]).strip() not in ("", "nan") else _slug(name)
        if sid in seen:
            continue
        pats = [_to_regex(x) for x in re.split(r"[|;]", str(r[c_pat]))] if c_pat and str(r[c_pat]) != "nan" else []
        pats = [x for x in pats if x] or [_to_regex(name)]
        for x in pats:
            re.compile(x)                                    # pola rusak langsung ketahuan
        new_skills.append((sid, name, str(r[c_cat]).strip() if c_cat else "Other", pats))
        seen.add(sid)
    SKILLS = new_skills
    print(f"D6 dipakai: {len(SKILLS)} skill, {len({s[2] for s in SKILLS})} kategori")

SKILL_IDS = {s[0] for s in SKILLS}
SKILL_NAME = {s[0]: s[1] for s in SKILLS}
_by_name = {s[1].strip().lower(): s[0] for s in SKILLS} | {s[0]: s[0] for s in SKILLS}

d7 = _read_table(D7_MODE, D7_DRIVE_PATH, "D7 peta mata kuliah–skill")
if d7 is not None:
    c_course, c_skill = _col(d7, "course", "mata_kuliah", "matakuliah", "course_name", "nama_mk", "mk"), _col(d7, "skill", "skill_id", "nama_skill")
    c_en, c_ev = _col(d7, "course_en", "nama_en", "english"), _col(d7, "evidence", "bukti", "evidence_text", "kalimat_bukti")
    assert c_course and c_skill, f"Kolom mata kuliah/skill tidak ditemukan. Kolom D7: {list(d7.columns)}"
    courses, unknown, evidence = {}, [], {}
    for _, r in d7.iterrows():
        cname, sk = str(r[c_course]).strip(), str(r[c_skill]).strip()
        if not cname or cname == "nan" or not sk or sk == "nan":
            continue
        sid = _by_name.get(sk.lower()) or _by_name.get(_slug(sk))
        if not sid:
            unknown.append((cname, sk)); continue
        cid = _slug(cname)
        c = courses.setdefault(cid, {"name_id": cname, "name_en": str(r[c_en]).strip() if c_en and str(r[c_en]) != "nan" else cname, "skills": []})
        if sid not in c["skills"]:
            c["skills"].append(sid)
        if c_ev and str(r[c_ev]) != "nan":
            evidence[(cid, sid)] = str(r[c_ev]).strip()
    COURSES = [(cid, c["name_id"], c["name_en"], c["skills"]) for cid, c in courses.items()]
    COURSE_EVIDENCE = evidence
    print(f"D7 dipakai: {sum(len(c[3]) for c in COURSES)} pasangan, {len(COURSES)} mata kuliah")
    if unknown:
        print(f"⚠️  {len(unknown)} pasangan dilewati karena skill-nya tidak ada di kamus:")
        for cname, sk in unknown[:15]:
            print(f"   - {cname} → '{sk}'")

if D6_MODE != "contoh" and D7_MODE == "contoh":
    COURSES = [(c[0], c[1], c[2], [s for s in c[3] if s in SKILL_IDS]) for c in COURSES]
    print("⚠️  D7 masih contoh; skill contoh yang tidak ada di D6 dibuang dari daftar mata kuliah.")

In [ ]:
SKILL_IDS = {s[0] for s in SKILLS}
SKILL_NAME = {s[0]: s[1] for s in SKILLS}
CAREER_IDS = [c[0] for c in CAREERS]
CAREER_NAME = {c[0]: c[1] for c in CAREERS}
for c in COURSES:
    assert set(c[3]) <= SKILL_IDS, f"skill tidak dikenal di mata kuliah {c[0]}"
print(f"{len(SKILLS)} skill, {len(CAREERS)} karier, {len(INTERESTS)} bidang minat, {len(COURSES)} mata kuliah")

## 3. Muat dataset lowongan

Pilih salah satu `DATA_MODE`:

- `"upload"` — unggah satu atau beberapa file dari komputer (`.csv`, `.xlsx`, atau `.zip` berisi CSV/Excel).
- `"drive"` — baca CSV dari Google Drive (isi `DRIVE_FILES`).
- `"demo"` — **hanya untuk mencoba notebook**: membuat lowongan sintetis dari angka awal. Hasil evaluasinya tidak bermakna dan **tidak boleh dilaporkan** di skripsi.

Kolom judul dan deskripsi dideteksi otomatis. Jika gagal, isi `COLUMN_MAP` per file, misalnya `{"jobs.csv": ("job_title", "job_description")}`.

Dataset di Tabel 3.4: JobStreet Indonesia 2021–2022 (Kaggle, hanya judul → tidak dipakai untuk permintaan), Indonesia job vacancy dataset (GitHub simaremare, CC0), Job Description and Salary in Indonesia (Kaggle canggih), Glints dan JobStreet 2025 (Zenodo, Zainuddin).

In [ ]:
DATA_MODE = "upload"     # "upload" | "drive" | "demo"
DRIVE_FILES = [          # dipakai jika DATA_MODE = "drive"
    # "/content/drive/MyDrive/skripsi/data/simaremare_jobs.csv",
]
COLUMN_MAP = {}          # {"nama_file.csv": ("kolom_judul", "kolom_deskripsi")}
DATA_LABEL = "dataset lowongan (isi nama & periode, mis. 'lowongan IT Indonesia 2015–2025')"  # tampil di website

TITLE_CANDIDATES = ["job_opening_title", "title", "job_title", "jobtitle", "position", "job_name", "judul", "posisi", "nama_pekerjaan", "role"]
DESC_CANDIDATES = ["description", "job_description", "jobdescription", "descriptions", "desc", "deskripsi",
                   "requirements", "requirement", "qualifications", "job_detail", "details", "kualifikasi"]

def pick_col(cols, candidates):
    low = {c.lower().strip().replace(" ", "_"): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    for cand in candidates:                       # cocok sebagian
        for k, orig in low.items():
            if cand in k:
                return orig
    return None

def load_frame(name, df):
    if name in COLUMN_MAP:
        tcol, dcol = COLUMN_MAP[name]
    else:
        tcol, dcol = pick_col(df.columns, TITLE_CANDIDATES), pick_col(df.columns, DESC_CANDIDATES)
    print(f"{name}: {len(df):,} baris | judul='{tcol}' | deskripsi='{dcol}'")
    if tcol is None:
        raise ValueError(f"Kolom judul tidak ditemukan di {name}. Kolom: {list(df.columns)} -> isi COLUMN_MAP")
    out = pd.DataFrame({"title": df[tcol].astype(str),
                        "description": df[dcol].astype(str) if dcol else ""})
    out["dataset"] = name
    return out

def read_csv_any(src):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(src, encoding=enc, on_bad_lines="skip", low_memory=False)
        except UnicodeDecodeError:
            if hasattr(src, "seek"):
                src.seek(0)
    raise ValueError("encoding tidak dikenali")

def read_any(name, data):
    # name: nama file; data: bytes. Mendukung .csv, .xlsx/.xls, dan .zip berisi file-file tersebut.
    # Mengembalikan list (nama, DataFrame).
    import zipfile
    low = name.lower()
    if low.endswith(".zip"):
        out = []
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for inner in z.namelist():
                if not inner.endswith("/") and not inner.startswith("__MACOSX") and inner.lower().endswith((".csv", ".xlsx", ".xls")):
                    out += read_any(os.path.basename(inner), z.read(inner))
        return out
    if low.endswith((".xlsx", ".xls")):
        return [(name, pd.read_excel(io.BytesIO(data)))]
    return [(name, read_csv_any(io.BytesIO(data)))]

In [ ]:
SEED_DEMAND = {
    "senior_programmer": {  # derived from SWE row of Table 2.1 + senior emphasis (estimate)
        "sql": .52, "java": .45, "javascript": .42, "php": .33, "html_css": .34, "git": .38,
        "rest_api": .36, "microservices": .25, "oop": .22, "spring": .18, "nodejs": .20,
        "android": .14, "agile_scrum": .24, "docker": .18, "mysql": .25, "unit_testing": .15,
        "react": .20, "golang": .12, "kotlin": .11,
    },
    "software_developer": {  # Table 2.1: SQL 49.5, JS 41.6, Java 40.7, HTML/CSS 40.4, PHP 35.5
        "sql": .495, "javascript": .416, "java": .407, "html_css": .404, "php": .355,
        "mysql": .28, "laravel": .20, "git": .24, "rest_api": .26, "react": .17, "codeigniter": .13,
        "python": .16, "nodejs": .15, "oop": .14, "csharp": .12, "dotnet": .12, "css_framework": .11,
        "vue": .10,
    },
    "system_analyst": {  # Table 2.1: Req. analysis 46.9, SQL 31.2, SDLC 21.9, Agile 15.1, UML 14.8
        "requirement_analysis": .469, "sql": .312, "sdlc": .219, "agile_scrum": .151, "uml": .148,
        "documentation": .28, "bpmn": .22, "erp": .12, "jira": .11, "data_modeling": .13,
    },
    "database_administrator": {  # Table 2.1: SQL 64.1, Backup 47.7, SQL Server 39.9, Oracle 28.8, PG 26.8
        "sql": .641, "backup_recovery": .477, "sql_server": .399, "oracle_db": .288,
        "postgresql": .268, "mysql": .35, "db_tuning": .30, "linux": .22, "data_modeling": .14,
        "mongodb": .13, "scripting": .12, "monitoring": .10,
    },
    # ---- placeholder estimates below: replace with measured demand ----
    "mobile_developer": {
        "android": .62, "kotlin": .42, "java": .38, "flutter": .35, "dart": .22, "ios": .28,
        "swift": .22, "react_native": .14, "rest_api": .32, "git": .28,
        "nosql": .14, "oop": .10,
    },
    "qa_tester": {
        "manual_testing": .48, "test_case": .45, "automation_testing": .40, "selenium": .32,
        "api_testing": .28, "sql": .24, "agile_scrum": .22, "jira": .20, "performance_testing": .14,
        "java": .12, "python": .12, "sdlc": .14, "javascript": .10,
    },
    "ui_ux_designer": {
        "figma": .62, "ui_design": .55, "ux_research": .45, "prototyping": .42, "adobe": .34,
        "design_system": .18, "html_css": .16, "agile_scrum": .10,
    },
    "data_analyst": {
        "sql": .62, "excel": .48, "python": .40, "power_bi": .30, "tableau": .28,
        "data_visualization": .32, "statistics": .30, "looker": .14, "pandas": .14, "r_lang": .12,
        "data_warehouse": .10,
    },
    "data_engineer": {
        "sql": .68, "python": .62, "etl": .48, "data_warehouse": .38, "spark": .32, "airflow": .26,
        "hadoop": .22, "kafka": .20, "gcp": .22, "aws": .20, "postgresql": .16,
        "docker": .14, "nosql": .14,
    },
    "data_scientist": {
        "python": .78, "machine_learning": .66, "sql": .48, "statistics": .40, "deep_learning": .30,
        "tensorflow": .24, "pytorch": .20, "scikit_learn": .22, "pandas": .26, "nlp": .16,
        "computer_vision": .12, "gen_ai": .12, "r_lang": .14, "spark": .10,
    },
    "devops_cloud": {
        "linux": .58, "docker": .55, "kubernetes": .46, "ci_cd": .45, "aws": .40, "gcp": .26,
        "azure": .20, "terraform": .24, "jenkins": .22, "ansible": .18, "monitoring": .30,
        "scripting": .32, "git": .28, "python": .18, "networking": .16,
    },
    "network_sysadmin": {
        "networking": .66, "linux": .44, "windows_server": .32, "cisco": .34, "mikrotik": .22,
        "firewall": .26, "virtualization": .24, "active_directory": .18, "troubleshooting": .30,
        "monitoring": .18, "network_security": .14, "scripting": .12,
    },
    "cyber_security": {
        "network_security": .52, "vuln_assessment": .42, "pentest": .36, "siem": .32,
        "firewall": .30, "linux": .32, "iso27001": .26, "networking": .30, "python": .14,
        "scripting": .12,
    },
    "it_support": {
        "troubleshooting": .58, "hardware": .48, "networking": .40, "os_install": .40,
        "ms_office": .30, "helpdesk": .28, "active_directory": .14, "windows_server": .14,
        "itil": .10,
    },
    "it_pm": {
        "project_management": .58, "agile_scrum": .52, "sdlc": .30, "jira": .30,
        "requirement_analysis": .28, "product_management": .26, "documentation": .18,
        "itil": .12, "erp": .10,
    },
}

def make_demo_ads(n_per_career=300, seed=7):
    """Lowongan SINTETIS dari SEED_DEMAND, hanya untuk mencoba notebook."""
    rng = random.Random(seed)
    titles = {
        "senior_programmer": ["Senior Software Engineer", "Lead Developer", "Sr. Programmer"],
        "software_developer": ["Web Developer", "Backend Developer", "Programmer", "Full Stack Developer"],
        "system_analyst": ["System Analyst", "IT Analyst"], "database_administrator": ["Database Administrator", "DBA"],
        "mobile_developer": ["Android Developer", "Mobile Developer"], "qa_tester": ["QA Engineer", "Software Tester"],
        "ui_ux_designer": ["UI/UX Designer", "Product Designer"], "data_analyst": ["Data Analyst", "Business Intelligence Analyst"],
        "data_engineer": ["Data Engineer"], "data_scientist": ["Data Scientist", "Machine Learning Engineer"],
        "devops_cloud": ["DevOps Engineer", "Cloud Engineer"], "network_sysadmin": ["Network Engineer", "System Administrator"],
        "cyber_security": ["Security Analyst", "SOC Analyst"], "it_support": ["IT Support", "Helpdesk Staff"],
        "it_pm": ["IT Project Manager", "Product Manager"],
    }
    rows = []
    for c, ts in titles.items():
        for i in range(n_per_career):
            chosen = [s for s, d in SEED_DEMAND[c].items() if s in SKILL_IDS and rng.random() < min(1, d * 1.3)]
            if rng.random() < .3:
                chosen.append(rng.choice(sorted(SKILL_IDS)))
            rows.append({"title": rng.choice(ts),
                         "description": f"Lowongan #{i}. Requirements: " + ", ".join(SKILL_NAME[s] for s in chosen) + ". Good communication and teamwork skills.",
                         "dataset": "DEMO_SINTETIS"})
    return pd.DataFrame(rows)

In [ ]:
frames = []
if DATA_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        for inner, df in read_any(name, content):
            frames.append(load_frame(inner, df))
elif DATA_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for path in DRIVE_FILES:
        for inner, df in read_any(os.path.basename(path), open(path, "rb").read()):
            frames.append(load_frame(inner, df))
elif DATA_MODE == "demo":
    print("⚠️  MODE DEMO: data sintetis. Jangan laporkan hasilnya.")
    frames.append(make_demo_ads())
else:
    raise ValueError("DATA_MODE harus 'upload', 'drive', atau 'demo'")

ads = pd.concat(frames, ignore_index=True)
ads["title"] = ads["title"].fillna("").str.strip()
ads["description"] = ads["description"].fillna("").replace("nan", "").str.strip()
before = len(ads)
ads = (ads.assign(_k=ads["title"].str.lower().str.strip() + "|" + ads["description"].str.lower().str[:300])
          .drop_duplicates("_k").drop(columns="_k").reset_index(drop=True))
print(f"Total {before:,} baris, {len(ads):,} setelah duplikat dihapus")
ads.head()

## 4. Klasifikasi judul lowongan ke 15 karier

Urutan pengecekan penting: peran spesifik dicek dulu, Senior Programmer sebelum Software Developer. Judul yang memuat kata non-IT (`NON_IT_TITLE`, misalnya *supplier*, *mechanical*, *sales*) langsung dibuang, begitu juga lowongan yang tidak cocok dengan karier IT mana pun.

In [ ]:
PRIORITY = ["mobile_developer", "data_engineer", "data_scientist", "data_analyst", "database_administrator",
            "devops_cloud", "cyber_security", "qa_tester", "ui_ux_designer", "system_analyst",
            "network_sysadmin", "it_support", "it_pm", "senior_programmer", "software_developer"]
_title_rules = [(cid, [re.compile(p, re.I) for p in next(c[4] for c in CAREERS if c[0] == cid)]) for cid in PRIORITY]

_non_it = [re.compile(p, re.I) for p in NON_IT_TITLE]

def classify_title(title):
    t = (title or "").lower()
    if any(p.search(t) for p in _non_it):   # judul non-IT (manufaktur, sales, logistik, ...)
        return None
    for cid, pats in _title_rules:
        if any(p.search(t) for p in pats):
            return cid
    return None

ads["career"] = ads["title"].map(classify_title)
it_ads = ads[ads["career"].notna()].copy()
it_ads["has_desc"] = it_ads["description"].str.len() >= 50
print(f"Lowongan IT terklasifikasi: {len(it_ads):,} dari {len(ads):,}")
summary = (it_ads.groupby("career").agg(semua=("title", "size"), dengan_deskripsi=("has_desc", "sum"))
           .reindex(CAREER_IDS).fillna(0).astype(int))
summary.index = summary.index.map(CAREER_NAME)
summary

**Validasi klasifikasi (3.2.5).** File di bawah berisi 20 lowongan acak per karier. Periksa secara manual, isi kolom `benar` dengan 1/0, lalu hitung akurasinya. Jika akurasi suatu karier < 80%, perbaiki pola judulnya di bagian 2 dan jalankan ulang.

In [ ]:
val_cls = (it_ads.sample(frac=1, random_state=SEED).groupby("career").head(20)
           .sort_values("career")[["career", "title", "dataset"]].copy())
val_cls["benar"] = ""
val_cls.to_csv(f"{OUT_DIR}/validasi_klasifikasi.csv", index=False)
print(f"{len(val_cls)} baris -> {OUT_DIR}/validasi_klasifikasi.csv")
val_cls.head(10)

## 5. Ekstraksi skill dari deskripsi

Hanya lowongan dengan deskripsi lengkap yang dipakai untuk mengukur permintaan.

In [ ]:
_skill_rules = [(sid, [re.compile(p, re.I) for p in pats]) for sid, _, _, pats in SKILLS]

def extract_skills(text):
    t = (text or "").lower()
    return sorted({sid for sid, pats in _skill_rules if any(p.search(t) for p in pats)})

def extract_with_phrase(text):
    t = (text or "").lower(); out = []
    for sid, pats in _skill_rules:
        for p in pats:
            m = p.search(t)
            if m:
                out.append((sid, t[max(0, m.start() - 40): m.end() + 40]))
                break
    return out

full = it_ads[it_ads["has_desc"]].copy()
full["skills"] = full["description"].map(extract_skills)
full["n_skills"] = full["skills"].map(len)
print(f"Lowongan dengan deskripsi lengkap: {len(full):,}")
print(f"Rata-rata skill per lowongan: {full['n_skills'].mean():.1f}")
top = Counter(s for ss in full["skills"] for s in ss).most_common(15)
pd.DataFrame([(SKILL_NAME[s], n, n / len(full)) for s, n in top], columns=["skill", "lowongan", "proporsi"])

**Validasi ekstraksi (3.2.5).** 50 lowongan acak beserta skill dan frasa pendukungnya. Tandai skill yang salah ekstrak untuk menghitung presisi.

In [ ]:
sample50 = full.sample(min(50, len(full)), random_state=SEED)
rows = [{"title": r.title, "skill": SKILL_NAME[s], "frasa": ph, "benar": ""}
        for r in sample50.itertuples() for s, ph in extract_with_phrase(r.description)]
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/validasi_ekstraksi.csv", index=False)
print(f"{len(rows)} pasangan lowongan–skill -> {OUT_DIR}/validasi_ekstraksi.csv")

## 6. Split 70/30 bertingkat per karier

Tidak ada model yang dilatih pada bagian 70%; split ini hanya memastikan lowongan uji tidak ikut membentuk profil permintaan.

In [ ]:
train_parts, test_parts = [], []
for c, g in full.groupby("career"):
    g = g.sample(frac=1, random_state=SEED)
    k = round(len(g) * TEST_FRAC)
    test_parts.append(g.iloc[:k]); train_parts.append(g.iloc[k:])
train, test = pd.concat(train_parts), pd.concat(test_parts)
split = pd.DataFrame({"latih (70%)": train.groupby("career").size(), "uji (30%)": test.groupby("career").size(),
                      "uji ≥3 skill": test[test.n_skills >= MIN_SKILLS_TEST].groupby("career").size()}
                     ).reindex(CAREER_IDS).fillna(0).astype(int)
split.index = split.index.map(CAREER_NAME)
print(f"Latih {len(train):,} | Uji {len(test):,} | Uji dengan ≥{MIN_SKILLS_TEST} skill: {(test.n_skills >= MIN_SKILLS_TEST).sum():,}")
split

**Profil uji (3.6.2).** Lowongan uji dengan kurang dari 3 skill dari kamus **diganti** dengan lowongan uji acak lain dari karier yang sama yang punya ≥3 skill, sehingga jumlah profil per karier tetap. Pengganti diambil dari bagian uji saja (tidak pernah dari bagian latih), dengan seed tetap. Karena diambil dengan pengembalian, satu lowongan bisa muncul lebih dari sekali; jumlahnya dilaporkan di bawah.

In [ ]:
rng_rep = np.random.default_rng(SEED)
parts, rep_report = [], []
for c, g in test.groupby("career"):
    ok, short = g[g.n_skills >= MIN_SKILLS_TEST], g[g.n_skills < MIN_SKILLS_TEST]
    if len(ok) == 0:
        rep_report.append({"karier": CAREER_NAME[c], "uji": len(g), "≥3 skill": 0, "diganti": 0, "catatan": "tidak ada pengganti → karier tidak diuji"})
        continue
    extra = ok.iloc[rng_rep.integers(0, len(ok), size=len(short))] if len(short) else ok.iloc[0:0]
    parts += [ok, extra]
    rep_report.append({"karier": CAREER_NAME[c], "uji": len(g), "≥3 skill": len(ok), "diganti": len(short), "catatan": ""})
test_profiles = pd.concat(parts, ignore_index=True)
print(f"Profil uji: {len(test_profiles):,} (dari {len(test):,} lowongan uji; {sum(r['diganti'] for r in rep_report):,} diganti)")
pd.DataFrame(rep_report)

## 7. Permintaan, himpunan kebutuhan, dan bobot

$$d(c,s)=\frac{n(c,s)}{N(c)} \qquad w(c,s)=d(c,s)\cdot\ln\left(1+\frac{|C|}{m(s)}\right)$$

$R(c)$ = skill dengan $d(c,s)\ge 10\%$. $m(s)$ = jumlah karier yang $R(c)$-nya memuat $s$, sehingga skill khas suatu karier berbobot lebih tinggi. Karier tanpa data latih memakai `SEED_DEMAND` dan ditandai.

In [ ]:
demand_rows = []
for c in CAREER_IDS:
    g = train[train.career == c]
    if len(g) == 0:
        print(f"⚠️  {CAREER_NAME[c]}: tidak ada lowongan latih -> memakai angka awal (estimasi)")
        demand_rows += [{"career": c, "skill": s, "demand": d, "n_ads": 0, "source": "seed"}
                        for s, d in SEED_DEMAND[c].items() if s in SKILL_IDS]
        continue
    cnt = Counter(s for ss in g.skills for s in ss)
    demand_rows += [{"career": c, "skill": s, "demand": n / len(g), "n_ads": len(g), "source": "measured"}
                    for s, n in cnt.items()]
demand = pd.DataFrame(demand_rows)

def build_requirements(demand_df, threshold=THRESHOLD):
    R = demand_df[demand_df.demand >= threshold].copy()
    m = R.groupby("skill").career.nunique()
    R["m"] = R.skill.map(m)
    R["weight"] = R.demand * np.log(1 + len(CAREER_IDS) / R.m)
    return R.sort_values(["career", "weight"], ascending=[True, False]).reset_index(drop=True)

REQ = build_requirements(demand)
demand.to_csv(f"{OUT_DIR}/demand.csv", index=False)
REQ.to_csv(f"{OUT_DIR}/requirements.csv", index=False)

tbl = (REQ.assign(item=lambda d: d.skill.map(SKILL_NAME) + " " + (d.demand * 100).round(1).astype(str) + "%")
          .sort_values(["career", "demand"], ascending=[True, False])
          .groupby("career").agg(n_ads=("n_ads", "first"), jumlah_skill=("skill", "size"),
                                 skill_teratas=("item", lambda x: "; ".join(x[:5]))))
tbl = tbl.reindex(CAREER_IDS); tbl.index = tbl.index.map(CAREER_NAME)
tbl

**Bobot skill per karier** (lima teratas). Skill yang diminta banyak karier, seperti SQL, mendapat pengali lebih kecil.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 10)); axes = axes.ravel()
for ax, c in zip(axes, CAREER_IDS):
    g = REQ[REQ.career == c].head(5)[::-1]
    ax.barh(g.skill.map(SKILL_NAME), g.weight, color="#1c2551")
    ax.set_title(CAREER_NAME[c], fontsize=10); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

## 7b. Triangulasi kebutuhan skill (3.2.2, validasi langkah 3)

Untuk setiap karier, dihitung proporsi skill di $R(c)$ yang juga muncul di minimal satu referensi independen:

1. **SKKNI dan peta okupasi**, berupa file `kb/chunks.jsonl` hasil `python build_rag.py --no-index` (sudah berisi karier per potongan), atau PDF/TXT dengan pemetaan karier di `SKKNI_FILE_CAREERS`.
2. **O\*NET Technology Skills**, berupa file `Technology Skills.txt` atau `.xlsx` dari onetcenter.org (database O\*NET). Okupasi O\*NET per karier diatur di `CAREER_ONET`.

Skill yang tidak muncul di referensi mana pun **wajib ditinjau manual** sebelum dipertahankan. Daftarnya disimpan di `triangulasi.csv`. Atur mode `"skip"` untuk melewati bagian ini.

In [ ]:
SKKNI_MODE = "skip"     # "skip" | "upload" | "drive"
SKKNI_DRIVE_FILES = []  # mis. ["/content/drive/MyDrive/skripsi/kb/chunks.jsonl"]
SKKNI_FILE_CAREERS = {  # hanya untuk PDF/TXT: nama file -> karier (file .jsonl sudah membawa karier sendiri)
    # "SKKNI_2016_282_Pemrograman.pdf": ["software_developer", "senior_programmer", "mobile_developer"],
}
ONET_MODE = "skip"      # "skip" | "upload" | "drive"
ONET_DRIVE_PATH = "/content/drive/MyDrive/skripsi/data/Technology Skills.txt"
CAREER_ONET = {         # karier -> kode O*NET-SOC (silakan sesuaikan)
    "senior_programmer": ["15-1252.00", "15-1254.00"],
    "software_developer": ["15-1252.00", "15-1254.00"],
    "system_analyst": ["15-1211.00"],
    "database_administrator": ["15-1242.00", "15-1243.00"],
    "mobile_developer": ["15-1252.00"],
    "qa_tester": ["15-1253.00"],
    "ui_ux_designer": ["15-1255.00"],
    "data_analyst": ["15-2051.01"],
    "data_engineer": ["15-1243.01", "15-1243.00"],
    "data_scientist": ["15-2051.00"],
    "devops_cloud": ["15-1299.08", "15-1244.00"],
    "network_sysadmin": ["15-1244.00", "15-1241.00", "15-1231.00"],
    "cyber_security": ["15-1212.00"],
    "it_support": ["15-1232.00", "15-1231.00"],
    "it_pm": ["15-1299.09"],
}

def _load_files(mode, drive_paths, label):
    if mode == "skip":
        return []
    if mode == "upload":
        from google.colab import files
        print(f"Unggah {label}:")
        return list(files.upload().items())
    if mode == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        return [(os.path.basename(p), open(p, "rb").read()) for p in drive_paths]
    raise ValueError("mode harus 'skip', 'upload', atau 'drive'")

def _pdf_text(data):
    try:
        from pypdf import PdfReader
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pypdf"], check=True)
        from pypdf import PdfReader
    return "\n".join((p.extract_text() or "") for p in PdfReader(io.BytesIO(data)).pages)

# 1) teks SKKNI/peta okupasi per karier
ref_text = defaultdict(str)
for name, data in _load_files(SKKNI_MODE, SKKNI_DRIVE_FILES, "SKKNI / peta okupasi (chunks.jsonl, PDF, atau TXT)"):
    if name.lower().endswith(".jsonl"):
        for line in data.decode("utf-8").splitlines():
            if line.strip():
                ch = json.loads(line)
                if ch.get("doc_type") in ("skkni", "occupation_map"):
                    for c in ch.get("careers", []):
                        ref_text[c] += "\n" + ch["text"]
    else:
        text = _pdf_text(data) if name.lower().endswith(".pdf") else data.decode("utf-8", "ignore")
        targets = SKKNI_FILE_CAREERS.get(name)
        if not targets:
            print(f"⚠️  {name} tidak ada di SKKNI_FILE_CAREERS → dipakai untuk semua karier")
            targets = CAREER_IDS
        for c in targets:
            ref_text[c] += "\n" + text
skkni_skills = {c: set(extract_skills(t)) for c, t in ref_text.items()}

# 2) O*NET Technology Skills per karier
onet_skills = {}
onet_files = _load_files(ONET_MODE, [ONET_DRIVE_PATH], "O*NET Technology Skills (.txt / .xlsx)")
if onet_files:
    name, data = onet_files[0]
    ot = pd.read_excel(io.BytesIO(data)) if name.lower().endswith((".xlsx", ".xls")) else pd.read_csv(io.BytesIO(data), sep="\t")
    code_col = next(c for c in ot.columns if "soc" in c.lower())
    ex_col = next(c for c in ot.columns if c.lower() in ("example", "technology skill", "example_name"))
    by_code = ot.groupby(code_col)[ex_col].apply(lambda x: " ; ".join(map(str, x)))
    for c, codes in CAREER_ONET.items():
        onet_skills[c] = set(extract_skills(" ; ".join(by_code.get(k, "") for k in codes)))
    print(f"O*NET: {len(ot):,} baris teknologi dibaca")

if skkni_skills or onet_skills:
    rows = []
    for c in CAREER_IDS:
        R = REQ[REQ.career == c].skill.tolist()
        if not R:
            continue
        sk, on = skkni_skills.get(c, set()), onet_skills.get(c, set())
        neither = [s for s in R if s not in sk and s not in on]
        rows.append({"karier": CAREER_NAME[c], "|R(c)|": len(R),
                     "di SKKNI/okupasi": sum(s in sk for s in R) / len(R) if skkni_skills else np.nan,
                     "di O*NET": sum(s in on for s in R) / len(R) if onet_skills else np.nan,
                     "di minimal satu": 1 - len(neither) / len(R),
                     "tinjau manual": ", ".join(SKILL_NAME[s] for s in neither)})
    triangulation = pd.DataFrame(rows)
    triangulation.to_csv(f"{OUT_DIR}/triangulasi.csv", index=False)
    display(triangulation.style.format({"di SKKNI/okupasi": "{:.0%}", "di O*NET": "{:.0%}", "di minimal satu": "{:.0%}"}))
else:
    print("Triangulasi dilewati (SKKNI_MODE dan ONET_MODE = 'skip').")

## 8. Cakupan kurikulum berbobot permintaan (RQ1)

Cakupan karier = jumlah bobot permintaan skill di $R(c)$ yang diajarkan minimal satu mata kuliah ÷ total bobot permintaan $R(c)$.

In [ ]:
course_skills = {s for c in COURSES for s in c[3]}
cov = []
for c in CAREER_IDS:
    g = REQ[REQ.career == c]
    covered = g[g.skill.isin(course_skills)]
    cov.append({"karier": CAREER_NAME[c], "skill_dibutuhkan": len(g),
                "cakupan_berbobot": covered.demand.sum() / g.demand.sum() if len(g) else np.nan,
                "belum_diajarkan": ", ".join(g[~g.skill.isin(course_skills)].skill.map(SKILL_NAME).head(6))})
coverage = pd.DataFrame(cov)
coverage.to_csv(f"{OUT_DIR}/coverage.csv", index=False)
coverage.style.format({"cakupan_berbobot": "{:.1%}"})

## 9. Mekanisme skor

$$e(s)=\max(e_{course},e_{self},e_{exp}) \quad K(c)=\frac{\sum_{s\in R(c)} w(c,s)\,e(s)}{\sum_{s\in R(c)} w(c,s)} \quad \text{Fit}(c)=100\,[\alpha K(c)+(1-\alpha)M(c)]$$

Deterministik dan tanpa LLM. Fungsi di bawah identik dengan `scoring.py` yang dipakai backend FastAPI.

In [ ]:
REQ_BY_CAREER = {c: list(REQ[REQ.career == c][["skill", "demand", "weight"]].itertuples(index=False)) for c in CAREER_IDS}
CAREER_GROUP = {c[0]: c[3] for c in CAREERS}
COURSE_MAP = {c[0]: c for c in COURSES}
INTEREST_MAP = {i[0]: i[3] for i in INTERESTS}

def evidence(profile):
    ev = {}
    def put(s, v, src):
        if v > ev.get(s, (-1, None))[0]:
            ev[s] = (v, src)
    for cid, grade in (profile.get("courses") or {}).items():
        if cid in COURSE_MAP:
            for s in COURSE_MAP[cid][3]:
                put(s, GRADE_POINTS.get(grade.upper(), 0.0), "course")
    for s, lv in (profile.get("skills") or {}).items():
        put(s, SELF_LEVEL.get(lv, 0.0), "self")
    for s in profile.get("experience_skills") or []:
        put(s, 1.0, "experience")
    return ev

def score(profile, alpha=ALPHA, use_interest=True, req=None):
    req = req or REQ_BY_CAREER
    ev = evidence(profile)
    linked = {c for i in (profile.get("interests") or []) for c in INTEREST_MAP.get(i, [])}
    out = []
    for order, c in enumerate(CAREER_IDS):
        rs = req[c]; tot = sum(r.weight for r in rs) or 1.0
        k = sum(r.weight * ev.get(r.skill, (0,))[0] for r in rs) / tot
        m = 1.0 if c in linked else 0.0
        fit = 100 * (alpha * k + (1 - alpha) * m) if use_interest else 100 * k
        out.append({"career": c, "fit": fit, "K": k, "M": m, "order": order,
                    "gaps": [r.skill for r in rs if ev.get(r.skill, (0,))[0] == 0]})
    out.sort(key=lambda r: (-r["fit"], r["order"]))
    return out

def cosine_rank(skills, req=None):
    req = req or REQ_BY_CAREER
    res = []
    for order, c in enumerate(CAREER_IDS):
        R = {r.skill for r in req[c]}
        denom = math.sqrt(len(R)) * math.sqrt(len(skills)) or 1.0
        res.append((-(len(R & skills) / denom), order, c))
    return [c for _, _, c in sorted(res)]

def recommend(profile, alpha=ALPHA, n_alt=3):
    r = score(profile, alpha)
    return [x for x in r if CAREER_GROUP[x["career"]] == "primary"], [x for x in r if CAREER_GROUP[x["career"]] == "alternative"][:n_alt]

# NFR1: input sama -> peringkat sama
_p = {"skills": {"sql": "advanced", "python": "intermediate"}, "interests": ["data_ai"]}
assert [r["career"] for r in score(_p)] == [r["career"] for r in score(_p)]
print("Mekanisme skor siap.")

## 10. Evaluasi akurasi peringkat (RQ4)

Setiap lowongan uji menjadi profil dengan $e(s)=1$ untuk skill yang ditemukan. Karena lowongan tidak berisi minat, uji ini hanya mengukur $K(c)$. Baseline: cosine antara vektor biner skill profil dan $R(c)$.

**Kriteria penerimaan:** Hit@3 ≥ 0,80 dan MRR lebih tinggi daripada baseline (uji Wilcoxon satu sisi, p < 0,05).

In [ ]:
from scipy.stats import wilcoxon

ev_rows = []
for r in test_profiles.itertuples():
    sk = set(r.skills)
    ours = [x["career"] for x in score({"skills": {s: "advanced" for s in sk}}, use_interest=False)]
    base = cosine_rank(sk)
    ev_rows.append({"career": r.career, "rank_ours": ours.index(r.career) + 1,
                    "rank_base": base.index(r.career) + 1, "top1_ours": ours[0]})
E = pd.DataFrame(ev_rows)

def metrics(ranks):
    return pd.Series({"Hit@1": (ranks == 1).mean(), "Hit@3": (ranks <= 3).mean(), "MRR": (1 / ranks).mean(), "n": len(ranks)})

per_ours = E.groupby("career").rank_ours.apply(metrics).unstack()
per_base = E.groupby("career").rank_base.apply(metrics).unstack()
macro = pd.DataFrame({"K(c) berbobot permintaan": per_ours[["Hit@1", "Hit@3", "MRR"]].mean(),
                      "Baseline cosine": per_base[["Hit@1", "Hit@3", "MRR"]].mean()}).T

rr_o, rr_b = 1 / E.rank_ours, 1 / E.rank_base
p = wilcoxon(rr_o, rr_b, alternative="greater").pvalue if (rr_o != rr_b).any() else 1.0
accepted = macro.loc["K(c) berbobot permintaan", "Hit@3"] >= 0.80 and \
           macro.loc["K(c) berbobot permintaan", "MRR"] > macro.loc["Baseline cosine", "MRR"] and p < 0.05
print(f"Profil uji: {len(E):,}  |  Wilcoxon (MRR ours > baseline) p = {p:.4g}")
print("Kriteria penerimaan:", "TERPENUHI" if accepted else "TIDAK TERPENUHI")
macro.style.format("{:.3f}")

In [ ]:
per = pd.concat({"ours": per_ours, "baseline": per_base}, axis=1)
per.index = per.index.map(CAREER_NAME)
per.style.format("{:.3f}")

**Confusion matrix** (karier sebenarnya vs karier peringkat 1). Perhatikan kebingungan Senior Programmer ↔ Software Developer (Keterbatasan 2).

In [ ]:
cm = pd.crosstab(E.career, E.top1_ours).reindex(index=CAREER_IDS, columns=CAREER_IDS, fill_value=0)
cmn = cm.div(cm.sum(axis=1).replace(0, 1), axis=0)
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(15)); ax.set_yticks(range(15))
ax.set_xticklabels([CAREER_NAME[c] for c in CAREER_IDS], rotation=60, ha="right", fontsize=8)
ax.set_yticklabels([CAREER_NAME[c] for c in CAREER_IDS], fontsize=8)
for i in range(15):
    for j in range(15):
        if cm.iat[i, j]:
            ax.text(j, i, cm.iat[i, j], ha="center", va="center", fontsize=7, color="white" if cmn.iat[i, j] > .5 else "black")
ax.set_xlabel("Peringkat 1 (prediksi)"); ax.set_ylabel("Karier sebenarnya"); plt.colorbar(im, fraction=.04)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150); plt.show()
sp, sd = "senior_programmer", "software_developer"
print(f"Lowongan Senior Programmer yang diperingkat 1 sebagai Software Developer: {cm.loc[sp, sd]} / {cm.loc[sp].sum()}")
json.dump({"macro": macro.to_dict(), "p_value": float(p), "accepted": bool(accepted),
           "n_profiles": int(len(E))}, open(f"{OUT_DIR}/evaluation.json", "w"), indent=1)

## 11. Sensitivitas α dan ablasi (3.6.3)

Untuk skripsi, isi `PROFILES` dengan profil 23 peserta (ekspor dari log website, `GET /api/export`). Contoh di bawah hanya ilustrasi. Ablasi Hit@3 untuk kelompok skill dihitung pada lowongan uji di sel berikutnya.

In [ ]:
PROFILES = [
    {"courses": {"basis_data": "A", "web": "B", "rpl": "B"}, "skills": {"python": "intermediate", "figma": "beginner"},
     "experience_skills": ["laravel", "mysql"], "interests": ["software_dev", "design"]},
    {"courses": {"statistika": "A", "data_mining": "B"}, "skills": {"sql": "intermediate", "excel": "advanced"},
     "experience_skills": ["power_bi"], "interests": ["data_ai"]},
    {"skills": {"linux": "intermediate", "networking": "advanced"}, "experience_skills": ["mikrotik", "troubleshooting"],
     "interests": ["infra_cloud", "security"]},
]
top3 = lambda p, a=ALPHA: [r["career"] for r in score(p, a)][:3]
rows = []
for i, p in enumerate(PROFILES):
    base = top3(p); row = {"profil": i + 1, "top3 (α=0.5)": ", ".join(CAREER_NAME[c] for c in base)}
    for a in (0.3, 0.7, 0.9):
        row[f"overlap α={a}"] = f"{len(set(base) & set(top3(p, a)))}/3"
    for g, empty in [("courses", {}), ("skills", {}), ("experience_skills", []), ("interests", [])]:
        ranked = [r["career"] for r in score(dict(p, **{g: empty}))]
        row[f"tanpa {g}: rank top1"] = ranked.index(base[0]) + 1
    rows.append(row)
pd.DataFrame(rows)

In [ ]:
# Sensitivitas ambang permintaan: Hit@3 dan MRR jika THRESHOLD diubah
sens = []
for th in (0.05, 0.10, 0.15, 0.20):
    req_th = {c: list(g[["skill", "demand", "weight"]].itertuples(index=False))
              for c, g in build_requirements(demand, th).groupby("career")}
    req_th = {c: req_th.get(c, []) for c in CAREER_IDS}
    ranks = []
    for r in test_profiles.itertuples():
        order = [x["career"] for x in score({"skills": {s: "advanced" for s in r.skills}}, use_interest=False, req=req_th)]
        ranks.append(order.index(r.career) + 1)
    ranks = np.array(ranks)
    sens.append({"threshold": th, "Hit@3": (ranks <= 3).mean(), "MRR": (1 / ranks).mean()})
pd.DataFrame(sens).style.format({"Hit@3": "{:.3f}", "MRR": "{:.3f}"})

## 12. Coba satu profil

In [ ]:
profil = {
    "courses": {"basis_data": "A", "apsi": "B"},                 # id mata kuliah -> nilai A–E
    "skills": {"sql": "intermediate", "figma": "beginner"},     # id skill -> beginner | intermediate | advanced
    "experience_skills": extract_skills("Magang sebagai QA, menulis test case dan memakai Postman serta Jira."),
    "interests": ["qa", "systems_analysis"],                    # 1–3 dari id INTERESTS
}
print("Skill dari pengalaman:", [SKILL_NAME[s] for s in profil["experience_skills"]])
primary, alternative = recommend(profil)
show = lambda rs: pd.DataFrame([{"karier": CAREER_NAME[r["career"]], "fit": round(r["fit"], 1), "K": f"{r['K']:.0%}",
                                 "minat": int(r["M"]), "3 gap teratas": ", ".join(SKILL_NAME[s] for s in r["gaps"][:3])} for r in rs])
display(show(primary)); display(show(alternative))

## 13. Ekspor model untuk website dan backend

Salin `model.json` ke `backend/model.json`, lalu restart FastAPI. Website Next.js membaca model lewat `GET /api/model`, jadi tidak perlu disalin ke frontend. (`model.js` hanya untuk versi HTML statis.)

In [ ]:
measured_any = (demand.source == "measured").any()
model = {
    "version": 1,
    "demand_source": "demand_measured.csv" if measured_any and DATA_MODE != "demo" else f"{DATA_MODE}_data",
    "data_label": "DATA DEMO SINTETIS" if DATA_MODE == "demo" else DATA_LABEL,
    "n_ads_train": int(len(train)),
    "params": {"alpha": ALPHA, "threshold": THRESHOLD, "n_careers": len(CAREER_IDS),
               "grade_points": GRADE_POINTS, "self_level": SELF_LEVEL},
    "careers": [{"id": c[0], "name_en": c[1], "name_id": c[2], "group": c[3],
                 "requirements": [{"skill": r.skill, "demand": round(float(r.demand), 4), "weight": round(float(r.weight), 6)}
                                  for r in REQ_BY_CAREER[c[0]]]} for c in CAREERS],
    "interests": [{"id": i[0], "name_en": i[1], "name_id": i[2], "careers": i[3]} for i in INTERESTS],
    "skills": [{"id": s[0], "name": s[1], "category": s[2], "patterns": s[3]} for s in SKILLS],
    "courses": [{"id": c[0], "name_id": c[1], "name_en": c[2], "skills": c[3]} for c in COURSES],
}
json.dump(model, open(f"{OUT_DIR}/model.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
open(f"{OUT_DIR}/model.js", "w", encoding="utf-8").write("window.KARIRFIT_MODEL = " + json.dumps(model, ensure_ascii=False) + ";\n")
demand[demand.source == "measured"].to_csv(f"{OUT_DIR}/demand_measured.csv", index=False)
test_profiles.assign(skills=test_profiles.skills.map("|".join))[["career", "title", "skills"]].to_csv(f"{OUT_DIR}/heldout_ads.csv", index=False)
if DATA_MODE == "demo":
    print("⚠️  Model ini dari data DEMO sintetis — jangan dipakai untuk website produksi atau skripsi.")
print(sorted(os.listdir(OUT_DIR)))

In [ ]:
# Unduh semua keluaran sebagai zip (atau salin ke Drive)
import shutil
shutil.make_archive("karirfit_output", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("karirfit_output.zip")
except ImportError:
    print("Bukan di Colab: file ada di karirfit_output.zip")

## 14. (Opsional) Indeks RAG

Membangun indeks FAISS dari potongan teks basis pengetahuan dengan `intfloat/multilingual-e5-base` (768 dimensi, IndexFlatIP pada vektor ternormalisasi). Unggah file JSONL potongan SKKNI / peta okupasi dengan format:

```json
{"id": "SKKNI-282-2016-001", "text": "...", "source": "SKKNI 282/2016 unit J.620100.001.02",
 "doc_type": "skkni", "careers": ["software_developer"], "access": "public", "retrievable": true}
```

Potongan profil karier dari data permintaan dibuat otomatis. Salin `index.faiss` dan `chunks.json` ke `backend/kb/`.

In [ ]:
BUILD_RAG = False   # ubah ke True untuk membangun indeks

if BUILD_RAG:
    !pip -q install sentence-transformers faiss-cpu
    import faiss
    from sentence_transformers import SentenceTransformer
    chunks = [{"id": f"CP-{c['id']}", "source": "Profil karier dari data lowongan", "doc_type": "career_profile",
               "careers": [c["id"]], "access": "public", "retrievable": True,
               "text": f"Career profile: {c['name_en']} ({c['name_id']}). Skills requested in at least 10% of Indonesian job ads: "
                       + ", ".join(f"{SKILL_NAME[r['skill']]} ({r['demand']:.0%})" for r in c["requirements"])}
              for c in model["careers"]]
    from google.colab import files
    for name, content in files.upload().items():          # JSONL SKKNI / peta okupasi (boleh dilewati)
        chunks += [json.loads(l) for l in content.decode("utf-8").splitlines() if l.strip()]
    enc = SentenceTransformer("intfloat/multilingual-e5-base")
    vecs = enc.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True, batch_size=32, show_progress_bar=True)
    index = faiss.IndexFlatIP(vecs.shape[1]); index.add(vecs)
    os.makedirs(f"{OUT_DIR}/kb", exist_ok=True)
    faiss.write_index(index, f"{OUT_DIR}/kb/index.faiss")
    json.dump(chunks, open(f"{OUT_DIR}/kb/chunks.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(f"{len(chunks)} potongan diindeks -> {OUT_DIR}/kb/")
    # contoh pencarian
    q = enc.encode(["query: Data Analyst. SQL, Power BI, statistics"], normalize_embeddings=True)
    s, ids = index.search(q, 5)
    for sc, i in zip(s[0], ids[0]):
        print(f"{sc:.3f}  {chunks[i]['id']}  {chunks[i]['text'][:90]}")